# NIDS CNN-LSTM Autoencoder (Colab Ready - Python 3.12 Compatible)

This notebook runs the full pipeline: preprocess → train → evaluate (CIC + CSE).

**Folder layout expected in Google Drive:**
```
MyDrive/nids-cnn-lstm-autoencoder/
  data/raw/CIC-IDS2017/
  data/raw/CSE-CIC-IDS2018/
```

If your dataset is elsewhere, update the paths in `config.yaml`.

## Requirements & Compatibility

- **Python 3.12**: Fully tested and compatible with Google Colab's Python 3.12 runtime
- **TensorFlow**: Auto-installs TensorFlow 2.16+ (Python 3.12 compatible)
- **Package Manager**: Prefers `uv` for fast installation, with automatic fallback to `pip`

## Runtime Recommendations

- **Google Colab GPU (T4/V100)**: Highly recommended for training (50-100x faster)
- **Local Windows (DirectML)**: Default config uses CPU (`training.force_cpu: true`) for stability
- **CPU-only**: Works but significantly slower for training phase

## Getting Started

1. Mount Google Drive (cell 2)
2. Notebook will auto-detect project root
3. Install dependencies (Python 3.12-compatible versions)
4. Optional: Download datasets from Kaggle
5. Run pipeline: Preprocess → Train → Evaluate


In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('[SKIP] Bukan runtime Colab, mount Drive dilewati.')


In [ ]:
# Move to project root (robust local/colab)
import os
from pathlib import Path

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _is_project_root(p: Path) -> bool:
    return (p / 'requirements.txt').exists() and (p / 'scripts').exists()

def _find_project_root() -> Path:
    env_root = os.environ.get('NIDS_PROJECT_ROOT', '').strip()
    if env_root:
        p = Path(env_root)
        if _is_project_root(p):
            return p

    # quick local candidates
    candidates = [Path.cwd(), *Path.cwd().parents]

    # colab candidates
    mydrive = Path('/content/drive/MyDrive')
    expected = mydrive / 'nids-cnn-lstm-autoencoder'
    expected_colab_nb = mydrive / 'Colab Notebooks' / 'nids-cnn-lstm-autoencoder'
    candidates.extend([expected_colab_nb, expected])

    for c in candidates:
        if _is_project_root(c):
            return c

    # deep scan only in Colab MyDrive
    if _is_colab_runtime() and mydrive.exists():
        for req in mydrive.rglob('requirements.txt'):
            c = req.parent
            if _is_project_root(c):
                return c

    raise FileNotFoundError(
        'Project root not found. Set env NIDS_PROJECT_ROOT or place repo under /content/drive/MyDrive.'
    )

PROJECT_ROOT = _find_project_root().resolve()
os.chdir(PROJECT_ROOT)

# ensure working dirs exist
for rel in [
    'data/raw/CIC-IDS2017',
    'data/raw/CSE-CIC-IDS2018',
    'data/processed',
    'models',
    'results',
]:
    (PROJECT_ROOT / rel).mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('CWD:', Path.cwd())
print('CIC CSV count:', len(list((PROJECT_ROOT / 'data/raw/CIC-IDS2017').glob('*.csv'))))
print('CSE CSV count:', len(list((PROJECT_ROOT / 'data/raw/CSE-CIC-IDS2018').glob('*.csv'))))


In [ ]:
# Install dependencies (Colab-aware: Python 3.12 compatible)
import sys
import shutil
import subprocess
from pathlib import Path

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

# Validate Python version
py_version = sys.version_info
print(f'[INFO] Python {py_version.major}.{py_version.minor}.{py_version.micro}')

if py_version < (3, 9):
    raise RuntimeError(f'Python 3.9+ required, got {py_version.major}.{py_version.minor}')

req = Path('requirements.txt').resolve()
if not req.exists():
    raise FileNotFoundError(f'requirements.txt not found at {req}')

def _run(cmd, suppress_error=False):
    print('[CMD]', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 and not suppress_error:
        print(result.stdout)
        print(result.stderr)
        raise subprocess.CalledProcessError(result.returncode, cmd)
    return result

def _install_with_pip(req_path: Path):
    _run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
    _run([sys.executable, '-m', 'pip', 'install', '-r', str(req_path)])

def _install_with_uv(req_path: Path):
    _run(['uv', 'pip', 'install', '-r', str(req_path)])

is_colab = _is_colab_runtime()

# CRITICAL: In Colab, clean install TensorFlow + NumPy to avoid binary incompatibility
if is_colab:
    print('[INFO] Colab mode: Checking if clean install is needed...')
    
    # Check if we need to do clean install
    need_clean_install = False
    try:
        import tensorflow as tf
        import numpy as np
        # Check versions
        tf_version = tuple(map(int, tf.__version__.split('.')[:2]))
        np_version = tuple(map(int, np.__version__.split('.')[:2]))
        
        if tf_version < (2, 16) or np_version < (1, 23):
            print(f'[INFO] Current: TensorFlow {tf.__version__}, NumPy {np.__version__}')
            print('[INFO] Outdated versions detected, need clean install')
            need_clean_install = True
        else:
            print(f'[SUCCESS] Already installed: TensorFlow {tf.__version__} + NumPy {np.__version__}')
    except (ImportError, ValueError, AttributeError) as e:
        print(f'[INFO] Import check failed: {e}')
        need_clean_install = True
    
    if need_clean_install:
        print('[INFO] Performing clean installation...')
        print('[IMPORTANT] If you see binary incompatibility errors, please:')
        print('            Runtime → Restart session → Then re-run from Cell 1')
        print('')
        
        # Upgrade pip and setuptools
        _run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
        
        # Uninstall everything related
        print('[INFO] Removing old packages...')
        _run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'tensorflow', 'tensorflow-cpu', 'tensorflow-gpu', 'numpy'], suppress_error=True)
        
        # Clear pip cache
        print('[INFO] Clearing pip cache...')
        _run([sys.executable, '-m', 'pip', 'cache', 'purge'], suppress_error=True)
        
        # Install NumPy first with specific compatible version
        print('[INFO] Installing NumPy 1.26.4...')
        _run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'numpy==1.26.4'])
        
        # Install TensorFlow
        print('[INFO] Installing TensorFlow 2.18.0...')
        _run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'tensorflow==2.18.0'])
        
        # Critical: Restart Python kernel to clear cached modules
        print('')
        print('=' * 70)
        print('⚠️  INSTALLATION COMPLETE - KERNEL RESTART REQUIRED')
        print('=' * 70)
        print('Due to Python module caching, please RESTART the runtime:')
        print('1. Go to: Runtime → Restart session')
        print('2. Re-run cells 1-3 in order')
        print('3. Installation will be skipped (already installed)')
        print('=' * 70)
        print('')
        
        # Try to programmatically restart (Colab-specific)
        try:
            import os
            os.kill(os.getpid(), 9)
        except Exception:
            raise RuntimeError(
                '\n\n' + '='*70 + '\n' +
                'IMPORTANT: Please manually restart runtime:\n' +
                '  Runtime → Restart session\n' +
                'Then re-run cells 1-3.\n' +
                '='*70
            )
    
    # Now install other dependencies (skip tensorflow and numpy)
    print('[INFO] Installing remaining dependencies...')
    filtered = []
    for line in req.read_text(encoding='utf-8').splitlines():
        s = line.strip()
        if not s or s.startswith('#'):
            continue
        low = s.lower()
        
        # Skip tensorflow and numpy (already installed)
        if low.startswith('tensorflow') or low.startswith('numpy'):
            continue
            
        filtered.append(line)
    
    if filtered:
        effective_req = Path('/tmp/requirements_colab_other.txt')
        effective_req.write_text('\n'.join(filtered) + '\n', encoding='utf-8')
        print(f'[INFO] Installing {len(filtered)} packages...')
        
        # Try uv first for other packages
        used_uv = False
        if shutil.which('uv'):
            try:
                _install_with_uv(effective_req)
                used_uv = True
                print('[INFO] Other dependencies installed via uv')
            except subprocess.CalledProcessError:
                print('[WARN] uv failed, falling back to pip...')
        
        if not used_uv:
            _run([sys.executable, '-m', 'pip', 'install', '-r', str(effective_req)])
            print('[INFO] Other dependencies installed via pip')
    
    print('[SUCCESS] All dependencies installed successfully')
    
else:
    # Local mode: use original requirements.txt
    print('[INFO] Local mode: Installing from requirements.txt')
    used_uv = False
    if shutil.which('uv'):
        try:
            _install_with_uv(req)
            used_uv = True
            print('[INFO] Dependencies installed via uv')
        except subprocess.CalledProcessError:
            print('[WARN] uv install failed, falling back to pip...')
    
    if not used_uv:
        _install_with_pip(req)
        print('[INFO] Dependencies installed via pip')

# Final verification
try:
    import tensorflow as tf
    import numpy as np
    print(f'\n[FINAL CHECK]')
    print(f'  TensorFlow: {tf.__version__}')
    print(f'  NumPy: {np.__version__}')
    print(f'  CUDA Built: {tf.test.is_built_with_cuda()}')
except ImportError as e:
    raise RuntimeError(f'Dependency verification failed: {e}')


In [ ]:
# System info & GPU check
import sys
import time
import shlex
import subprocess
import platform

print('=' * 60)
print('SYSTEM INFORMATION')
print('=' * 60)
print(f'Python: {sys.version}')
print(f'Executable: {sys.executable}')
print(f'Platform: {platform.platform()}')
print(f'Processor: {platform.processor() or "Unknown"}')

try:
    import tensorflow as tf
    print(f'\nTensorFlow: {tf.__version__}')
    print(f'Keras: {tf.keras.__version__}')
    
    gpus = tf.config.list_physical_devices('GPU')
    print(f'\nGPU Devices: {len(gpus)}')
    if gpus:
        for i, gpu in enumerate(gpus):
            print(f'  [{i}] {gpu.name} ({gpu.device_type})')
            try:
                details = tf.config.experimental.get_device_details(gpu)
                if details:
                    print(f'      Compute Capability: {details.get("compute_capability", "N/A")}')
            except Exception:
                pass
    else:
        print('  No GPU detected - will use CPU')
    
    print(f'\nCUDA Built: {tf.test.is_built_with_cuda()}')
    if tf.test.is_built_with_cuda():
        print(f'CUDA Available: {tf.test.is_gpu_available(cuda_only=True) if hasattr(tf.test, "is_gpu_available") else "Check skipped"}')
except ImportError as e:
    print(f'\n[ERROR] TensorFlow not available: {e}')
    raise

print('=' * 60)

def _fmt_duration(seconds: float) -> str:
    sec = max(0, int(seconds))
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f'{h}h {m:02d}m {s:02d}s'
    if m > 0:
        return f'{m}m {s:02d}s'
    return f'{s}s'

def _normalize_cmd(cmd: str):
    parts = shlex.split(cmd, posix=False)
    if parts and parts[0].lower() == 'python':
        parts[0] = sys.executable
    return parts

def run_stage(stage_no: int, total_stages: int, title: str, commands):
    if isinstance(commands, str):
        commands = [commands]

    start_pct = ((stage_no - 1) / total_stages) * 100
    end_pct = (stage_no / total_stages) * 100
    print(f'\n[STAGE {stage_no}/{total_stages}] {title}')
    print(f'Global progress: {start_pct:.0f}% → {end_pct:.0f}%')
    print('-' * 60)

    t0 = time.time()
    for i, cmd in enumerate(commands, start=1):
        args = _normalize_cmd(cmd)
        print(f'[CMD {i}/{len(commands)}] {" ".join(args)}')
        result = subprocess.run(args, check=False, cwd=str(PROJECT_ROOT))
        if result.returncode != 0:
            raise RuntimeError(f'Command failed with exit code {result.returncode}')

    duration = _fmt_duration(time.time() - t0)
    print(f'[DONE] Stage {stage_no}/{total_stages} ({title}) completed in {duration}')
    print('=' * 60)


## Optional: Download Dataset from Kaggle

**Status**: Default OFF (`ENABLE_KAGGLE_DOWNLOAD = False`)

Activate this cell only if you need to download datasets directly from Kaggle.

### Requirements:
1. Kaggle API token (`kaggle.json`)
2. Upload token to one of these locations in Google Drive:
   - `/content/drive/MyDrive/kaggle.json`
   - `/content/drive/MyDrive/Colab Notebooks/kaggle.json`
   - `/content/drive/MyDrive/nids-cnn-lstm-autoencoder/kaggle.json`

### How to get kaggle.json:
1. Visit: https://www.kaggle.com/settings/account
2. Scroll to "API" section
3. Click "Create New Token"
4. Upload the downloaded `kaggle.json` to Google Drive

### Datasets:
- **CIC-IDS2017**: `chethuhn/network-intrusion-dataset`
- **CSE-CIC-IDS2018**: `solarmainframe/ids-intrusion-csv`

**Note**: Change `ENABLE_KAGGLE_DOWNLOAD = True` in the cell below to activate download.


In [ ]:
# Optional Kaggle downloader (Colab only)
from pathlib import Path
import sys
import os
import shutil
import subprocess
import zipfile

ENABLE_KAGGLE_DOWNLOAD = False  # ubah ke True kalau ingin download
OVERWRITE_EXISTING = True       # True kalau ingin timpa file lama

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _run(cmd):
    print('[CMD]', ' '.join(cmd))
    subprocess.run(cmd, check=True)

if not _is_colab_runtime():
    print('[SKIP] Bukan runtime Colab. Cell download Kaggle tidak dijalankan.')
elif not ENABLE_KAGGLE_DOWNLOAD:
    print('[SKIP] ENABLE_KAGGLE_DOWNLOAD=False. Aktifkan jika ingin download dataset.')
else:
    project_root = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path(os.getcwd())
    raw_cic = project_root / 'data' / 'raw' / 'CIC-IDS2017'
    raw_cse = project_root / 'data' / 'raw' / 'CSE-CIC-IDS2018'
    raw_cic.mkdir(parents=True, exist_ok=True)
    raw_cse.mkdir(parents=True, exist_ok=True)

    # Setup kaggle token - try multiple common locations
    possible_token_paths = [
        Path('/content/drive/MyDrive/kaggle.json'),
        Path('/content/drive/MyDrive/Colab Notebooks/kaggle.json'),
        Path('/content/drive/MyDrive/nids-cnn-lstm-autoencoder/kaggle.json'),
        project_root / 'kaggle.json',
    ]
    
    drive_token = None
    for token_path in possible_token_paths:
        if token_path.exists():
            drive_token = token_path
            print(f'[INFO] Found kaggle.json at {drive_token}')
            break
    
    if not drive_token:
        raise FileNotFoundError(
            f'kaggle.json not found. Tried locations:\n' +
            '\n'.join(f'  - {p}' for p in possible_token_paths) +
            '\n\nDownload from https://www.kaggle.com/settings/account and upload to Google Drive.'
        )

    _run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'])
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_target = kaggle_dir / 'kaggle.json'
    shutil.copy2(drive_token, token_target)
    os.chmod(token_target, 0o600)

    tmp = Path('/content/kaggle_tmp')
    tmp.mkdir(parents=True, exist_ok=True)

    def download_and_extract(dataset_slug: str, zip_name: str, out_dir: Path):
        zip_path = tmp / zip_name
        _run(['kaggle', 'datasets', 'download', '-d', dataset_slug, '-p', str(tmp), '--force'])
        if not zip_path.exists():
            # fallback: kaggle kadang pakai nama file berdasarkan slug
            candidates = sorted(tmp.glob('*.zip'), key=lambda x: x.stat().st_mtime, reverse=True)
            if not candidates:
                raise FileNotFoundError(f'Zip tidak ditemukan untuk {dataset_slug}')
            zip_path = candidates[0]
        extract_dir = tmp / (zip_path.stem + '_extract')
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_dir)

        csvs = sorted(extract_dir.rglob('*.csv'))
        copied = 0
        for f in csvs:
            dst = out_dir / f.name
            if dst.exists() and not OVERWRITE_EXISTING:
                continue
            shutil.copy2(f, dst)
            copied += 1
        print(f'[DONE] {dataset_slug}: copied {copied} CSV files to {out_dir}')

    # CIC-IDS2017 mirror (Kaggle)
    download_and_extract('chethuhn/network-intrusion-dataset', 'network-intrusion-dataset.zip', raw_cic)

    # CSE-CIC-IDS2018 mirror (Kaggle)
    download_and_extract('solarmainframe/ids-intrusion-csv', 'ids-intrusion-csv.zip', raw_cse)

    print('[SUMMARY] CIC CSV:', len(list(raw_cic.glob('*.csv'))))
    print('[SUMMARY] CSE CSV:', len(list(raw_cse.glob('*.csv'))))


## 1) Preprocess
Generates (default sharded mode):
- data/processed/shards/cic/train/*.npz + manifest.json
- data/processed/shards/cic/val/*.npz + manifest.json
- data/processed/shards/cic/test/*.npz + manifest.json
- data/processed/shards/cse/test/*.npz + manifest.json
- data/processed/scaler.pkl
- data/processed/feature_columns.json


In [ ]:
run_stage(1, 4, 'Preprocess', 'python scripts/preprocess.py --config config.yaml')


## 2) Train Hybrid CNN-LSTM Autoencoder

**IMPORTANT**: In Google Colab with GPU, the config will be auto-adjusted to use GPU (force_cpu → false).

Run the next cell first to optimize config for Colab environment.

### 💾 Checkpoint & Resume Features

**Automatic Checkpointing**:
- Best model saved to: `models/cnn_lstm_ae/best_model.keras` (based on validation loss)
- Periodic checkpoints every 5 epochs: `models/cnn_lstm_ae/checkpoints/checkpoint_epoch_XXX.keras`
- Training history saved to: `results/logs/cnn_lstm_history.json`

**Resume Training** (Colab disconnect recovery):
- If training is interrupted, simply **re-run the training cell**
- Script will automatically detect existing checkpoint and resume from last epoch
- No manual intervention needed

**Example Workflow**:
1. Start training → Epoch 1-20 complete
2. Colab disconnects (timeout/crash)
3. Restart runtime → Run cells 1-4 (mount, root, install, GPU config)
4. Re-run training cell → **Automatically resumes from epoch 20**


In [ ]:
# Auto-adjust config for Colab GPU (if available)
import yaml
from pathlib import Path

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _is_colab_runtime():
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    
    if gpus:
        print(f'[INFO] Detected {len(gpus)} GPU(s) in Colab')
        for i, gpu in enumerate(gpus):
            print(f'  [{i}] {gpu.name}')
        
        # Load config
        config_path = Path('config.yaml')
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        
        # Check current setting
        current_force_cpu = config.get('training', {}).get('force_cpu', True)
        
        if current_force_cpu:
            print(f'\n[ACTION] Overriding config: force_cpu: true → false')
            print('[INFO] Training will use GPU for 50-100x speedup')
            
            # Override force_cpu
            if 'training' not in config:
                config['training'] = {}
            config['training']['force_cpu'] = False
            
            # Save modified config
            with open(config_path, 'w') as f:
                yaml.dump(config, f, default_flow_style=False, sort_keys=False)
            
            print(f'[SUCCESS] Config updated: {config_path}')
            print(f'[VERIFY] force_cpu = {config["training"]["force_cpu"]}')
        else:
            print(f'[INFO] Config already optimized: force_cpu = false')
    else:
        print('[WARN] No GPU detected - will use CPU (slow)')
        print('[TIP] Change runtime: Runtime → Change runtime type → GPU (T4)')
else:
    print('[SKIP] Not running in Colab, using original config')


In [ ]:
# run_stage(2, 4, 'Train Hybrid CNN-LSTM AE', 'python scripts/train_cnn_lstm_ae.py --config config.yaml')

import sys, subprocess

cmd = [sys.executable, "scripts/train_cnn_lstm_ae.py", "--config", "config.yaml"]

proc = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

ret = proc.wait()
print(f"\n[EXIT CODE] {ret}")
if ret != 0:
    raise RuntimeError(f"Training failed. Last log line: {line.strip()}")

## 3) Evaluate (CIC + CSE)
Outputs metrics + plots in `results/` using the same threshold logic from config (percentile on CIC validation BENIGN).

Example outputs for `--tag cnn_lstm`:
- results/metrics/cnn_lstm_cic_metrics.json
- results/metrics/cnn_lstm_cse_metrics.json
- results/metrics/cnn_lstm_generalization_gap.json
- results/plots/cnn_lstm/roc_cic.png
- results/plots/cnn_lstm/roc_cse.png


In [ ]:
run_stage(3, 4, 'Evaluate (CIC + CSE)', 'python scripts/eval_metrics.py --config config.yaml --model models/cnn_lstm_ae/best_model.keras --tag cnn_lstm')


## 4) Optional: Train LSTM Autoencoder Baseline


In [ ]:
run_stage(4, 4, 'Baseline LSTM AE', [
    'python scripts/train_lstm_ae.py --config config.yaml',
    'python scripts/eval_metrics.py --config config.yaml --model models/lstm_ae/best_model.keras --tag lstm_ae',
])


## Troubleshooting & Notes

### Python 3.12 Compatibility

This notebook is fully compatible with Google Colab's Python 3.12 runtime:
- ✅ **TensorFlow 2.16+**: Auto-installed (supports Python 3.12)
- ✅ **NumPy 1.26+**: Compatible with TensorFlow 2.16+
- ✅ **All dependencies**: Tested on Python 3.12

### Common Issues

#### 1. **Project root not found**
**Error**: `FileNotFoundError: Project root not found`

**Solutions**:
- Ensure folder structure exists in Google Drive: `MyDrive/nids-cnn-lstm-autoencoder/`
- Check that `requirements.txt` and `scripts/` folder exist
- Set environment variable: `os.environ['NIDS_PROJECT_ROOT'] = '/path/to/project'`

#### 2. **TensorFlow import error**
**Error**: `ImportError: cannot import name 'tf'`

**Solutions**:
- Restart runtime: Runtime → Restart Runtime
- Manually install: `!pip install --upgrade tensorflow>=2.16.0`
- Check Python version: Should be 3.9+ (Colab uses 3.12)

#### 3. **Dataset not found**
**Error**: `CIC CSV count: 0` or `CSE CSV count: 0`

**Solutions**:
- Download datasets manually and upload to Drive
- Use Kaggle download cell (set `ENABLE_KAGGLE_DOWNLOAD = True`)
- Verify paths in `config.yaml` match your Drive structure

#### 4. **Out of memory during training**
**Error**: `ResourceExhaustedError` or `OOM`

**Solutions**:
- Use Google Colab Pro (more RAM)
- Reduce `batch_size` in `config.yaml` (default: 128 → try 64 or 32)
- Enable GPU: Runtime → Change runtime type → GPU (T4)
- Reduce `shard_size` in preprocessing (default: 20000 → try 10000)

#### 5. **Slow training on CPU**
**Issue**: Training takes hours

**Solutions**:
- **RECOMMENDED**: Use Google Colab with GPU runtime
  - Runtime → Change runtime type → GPU (T4 or V100)
  - Free tier: T4 (30x faster than CPU)
  - Pro tier: V100 (100x faster than CPU)
- In `config.yaml`, set `training.force_cpu: false`

### Performance Tips

1. **GPU Usage**: Always use GPU runtime in Colab for training
2. **Batch Size**: Larger batch = faster training (if memory allows)
3. **Sharding**: Keep `shard_enable: true` for large datasets
4. **Early Stopping**: Automatically saves best model (patience=10 epochs)

### Expected Runtime (Google Colab GPU T4)

- **Preprocessing**: 5-15 minutes (depending on dataset size)
- **Training CNN-LSTM**: 30-60 minutes (100 epochs with early stopping)
- **Evaluation**: 5-10 minutes
- **Total**: ~1-1.5 hours for complete pipeline

### Support & Documentation

- **Config**: See `config.yaml` for all parameters
- **Scripts**: Check `scripts/` folder for implementation details
- **Papers**: See `my-papers/` for research background
- **Docs**: Check `docs/` for technical guides

### Version Info

- **Notebook Version**: 2.0 (Python 3.12 compatible)
- **Last Updated**: 2026-02-15
- **Tested On**: Google Colab (Python 3.12, TensorFlow 2.16+)
